# Experiment 1: Partition Training Data (ID vs ID+OOD)

How OOD detection quality changes when partitioning schemes are trained on different data sources:
- `id`: partitions are fitted on ID train data only.
- `id+ood`: partitions are fitted on ID train and OOD train data.

Partition fitting uses only training splits. Test ID/OOD samples are never used when fitting partition schemes or the final classifier.

In [12]:
import pandas as pd
from IPython.display import display

from partition_ood import (
    DATASETS,
    load_dataset,
    cross_validate_ood_classifiers,
    plot_cv_metric_distributions,
    ALL_META_FEATURES,
    DEFAULT_META_FEATURES,
)

In [19]:
TEST_SIZE = 0.3
RANDOM_STATE = 42
N_BINS = 10
N_KMEANS_CLUSTERS = 10
N_TREE_PARTITIONS = 5
TREE_MAX_DEPTH = 3
USE_RAW_FEATURES = True
CV_FOLDS = 5
BALANCE_STRATEGY = 'oversample'
INCLUDE_MLP = True
PARTITION_FIT_MODES = ["id", "id+ood"]
DATASET_NAMES = list(DATASETS.keys())

TAXONOMY_META_FEATURES = [
    'dist_to_mean', 'normalized_dist', 'cosine_dist', 'dist_to_median', 'log_likelihood',
    'local_outlier_factor_score', 'knn_distance_k',
    'mean_norm', 'std_norm', 'skew_norm', 'kurtosis_norm', 'median_abs_deviation_norm',
    'iqr_norm', 'trimmed_mean_norm',
    'cell_entropy', 'marginal_entropy_mean', 'marginal_entropy_std',
    'marginal_kl_to_reference_mean', 'quantile_surprisal_mean',
    'mahalanobis', 'covariance_trace', 'covariance_logdet', 'mean_abs_correlation',
    'pairwise_pearson_corr_mean_abs', 'pairwise_spearman_corr_mean_abs',
    'covariance_condition_number',
    'pairwise_mutual_info_mean', 'pairwise_mutual_info_max', 'total_correlation',
    'joint_entropy_pairwise_mean', 'pairwise_js_divergence_mean',
    'log_count', 'density', 'out_of_range_count', 'n_beyond_2std',
    'partition_agreement_count', 'leaf_depth',
]
META_FEATURES = TAXONOMY_META_FEATURES
META_FEATURES

['log_count',
 'density',
 'mean_norm',
 'std_norm',
 'dist_to_mean',
 'mean_radius',
 'max_radius',
 'cell_entropy',
 'skew_norm',
 'kurtosis_norm']

In [20]:
def evaluate_dataset_mode(dataset_name, partition_fit_data):
    id_data, ood_data = load_dataset(dataset_name)
    X_train, X_test, y_train, y_test, model, repr_scaler = build_ood_dataset(
        id_data,
        ood_data,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
        n_bins=N_BINS,
        n_kmeans_clusters=N_KMEANS_CLUSTERS,
        n_tree_partitions=N_TREE_PARTITIONS,
        tree_max_depth=TREE_MAX_DEPTH,
        use_raw_features=USE_RAW_FEATURES,
        meta_features=META_FEATURES,
        partition_fit_data=partition_fit_data,
    )
    metrics_df, classifiers = train_ood_classifiers(
        X_train, X_test, y_train, y_test, dataset_name=dataset_name
    )
    tpr_df = tpr_at_fpr(classifiers, X_test, y_test)
    return metrics_df, tpr_df


def run_dataset_experiment(dataset_name):
    dataset_metrics = []
    dataset_tpr = []

    for mode in PARTITION_FIT_MODES:
        metrics_df, tpr_df = evaluate_dataset_mode(dataset_name, mode)

        metrics_tmp = metrics_df.copy().reset_index().rename(columns={"index": "classifier"})
        metrics_tmp.insert(0, "partition_fit_data", mode)
        metrics_tmp.insert(0, "dataset", dataset_name)
        dataset_metrics.append(metrics_tmp)

        tpr_tmp = tpr_df.copy().reset_index().rename(columns={"index": "classifier"})
        tpr_tmp.insert(0, "partition_fit_data", mode)
        tpr_tmp.insert(0, "dataset", dataset_name)
        dataset_tpr.append(tpr_tmp)

    dataset_metrics_df = pd.concat(dataset_metrics, axis=0, ignore_index=True)
    dataset_tpr_df = pd.concat(dataset_tpr, axis=0, ignore_index=True)

    display(dataset_metrics_df)
    display(dataset_tpr_df)

    return dataset_metrics_df, dataset_tpr_df


all_metrics = []
all_tpr = []

import matplotlib.pyplot as plt
import seaborn as sns

metrics_to_plot = ["roc-auc", "pr-auc", "accuracy", "precision", "recall", "f1-score"]


def plot_dataset_metric_barcharts(dataset_name, dataset_metrics_df):
    classifiers_order = sorted(dataset_metrics_df["classifier"].unique())

    for metric_name in metrics_to_plot:
        fig, axes = plt.subplots(1, len(classifiers_order), figsize=(8 * len(classifiers_order), 5), sharey=True)

        if len(classifiers_order) == 1:
            axes = [axes]

        for ax, clf_name in zip(axes, classifiers_order):
            plot_df = dataset_metrics_df[dataset_metrics_df["classifier"] == clf_name].copy()
            sns.barplot(
                data=plot_df,
                x="partition_fit_data",
                y=metric_name,
                hue="partition_fit_data",
                hue_order=["id", "id+ood"],
                order=["id", "id+ood"],
                dodge=False,
                ax=ax,
            )
            ax.set_title(f"{dataset_name} | {clf_name}")
            ax.set_xlabel("Partition fit data")
            ax.set_ylabel(metric_name)

        fig.suptitle(f"{dataset_name}: ID vs ID+OOD | {metric_name}", fontsize=14)
        plt.tight_layout()
        plt.show()

In [ ]:
def evaluate_dataset_mode(dataset_name, partition_fit_data):
    id_data, ood_data = load_dataset(dataset_name)
    fold_metrics, summary_df, _ = cross_validate_ood_classifiers(
        ID_data=id_data,
        OOD_data=ood_data,
        n_splits=CV_FOLDS,
        random_state=RANDOM_STATE,
        n_bins=N_BINS,
        n_kmeans_clusters=N_KMEANS_CLUSTERS,
        n_tree_partitions=N_TREE_PARTITIONS,
        tree_max_depth=TREE_MAX_DEPTH,
        use_raw_features=USE_RAW_FEATURES,
        meta_features=META_FEATURES,
        partition_fit_data=partition_fit_data,
        balance_strategy=BALANCE_STRATEGY,
        include_mlp=INCLUDE_MLP,
    )
    return fold_metrics, summary_df


def run_dataset_experiment(dataset_name):
    summary_parts = []
    fold_parts = []

    for mode in PARTITION_FIT_MODES:
        fold_metrics, summary_df = evaluate_dataset_mode(dataset_name, mode)
        fold_tmp = fold_metrics.copy()
        fold_tmp.insert(0, 'partition_fit_data', mode)
        fold_tmp.insert(0, 'dataset', dataset_name)
        fold_parts.append(fold_tmp)

        summary_tmp = summary_df.copy()
        summary_tmp.insert(0, 'partition_fit_data', mode)
        summary_tmp.insert(0, 'dataset', dataset_name)
        summary_parts.append(summary_tmp)

    dataset_summary_df = pd.concat(summary_parts, ignore_index=True)
    dataset_folds_df = pd.concat(fold_parts, ignore_index=True)

    display(dataset_summary_df)

    for metric_name in ['roc_auc_mean', 'pr_auc_mean', 'f1_mean']:
        plt.figure(figsize=(9, 5))
        sns.barplot(
            data=dataset_summary_df,
            x='model',
            y=metric_name,
            hue='partition_fit_data',
            hue_order=['id', 'id+ood'],
        )
        plt.title(f'{dataset_name}: CV {metric_name} by model')
        plt.xlabel('Model')
        plt.ylabel(metric_name)
        plt.xticks(rotation=10)
        plt.tight_layout()
        plt.show()

    return dataset_summary_df, dataset_folds_df


all_metrics = []
all_fold_metrics = []

import matplotlib.pyplot as plt
import seaborn as sns

## Taxi Data

In [22]:
taxi_metrics, taxi_tpr = run_dataset_experiment("Taxi")
all_metrics.append(taxi_metrics)
all_tpr.append(taxi_tpr)
#plot_dataset_metric_barcharts("Taxi", taxi_metrics)

,dataset,partition_fit_data,classifier,roc-auc,pr-auc,accuracy,precision,recall,f1-score
0,Taxi,id,Logistic Regression,0.9297,0.9221,0.8717,0.8632,0.8833,0.8731
1,Taxi,id,Random Forest,0.9746,0.9745,0.9095,0.9212,0.8957,0.9082
2,Taxi,id+ood,Logistic Regression,0.9997,0.9997,0.9907,0.9920,0.9893,0.9907
3,Taxi,id+ood,Random Forest,0.9996,0.9996,0.9920,0.9900,0.9940,0.9920


,dataset,partition_fit_data,classifier,0.01,0.05,0.1
0,Taxi,id,Logistic Regression,0.2493,0.6757,0.8333
1,Taxi,id,Random Forest,0.6600,0.8567,0.9250
2,Taxi,id+ood,Logistic Regression,0.9933,1.0000,1.0000
3,Taxi,id+ood,Random Forest,0.9940,0.9997,1.0000


## Electricity Data

In [23]:
electricity_metrics, electricity_tpr = run_dataset_experiment("Electricity")
all_metrics.append(electricity_metrics)
all_tpr.append(electricity_tpr)
#plot_dataset_metric_barcharts("Electricity", electricity_metrics)

,dataset,partition_fit_data,classifier,roc-auc,pr-auc,accuracy,precision,recall,f1-score
0,Electricity,id,Logistic Regression,0.9578,0.9466,0.8904,0.8671,0.9225,0.8939
1,Electricity,id,Random Forest,0.9810,0.9772,0.9280,0.8997,0.9637,0.9306
2,Electricity,id+ood,Logistic Regression,0.9592,0.9487,0.8870,0.8597,0.9255,0.8913
3,Electricity,id+ood,Random Forest,0.9818,0.9785,0.9255,0.8967,0.9621,0.9282


,dataset,partition_fit_data,classifier,0.01,0.05,0.1
0,Electricity,id,Logistic Regression,0.4186,0.7058,0.8532
1,Electricity,id,Random Forest,0.6366,0.8696,0.9567
2,Electricity,id+ood,Logistic Regression,0.3980,0.7195,0.8549
3,Electricity,id+ood,Random Forest,0.6326,0.8865,0.9557


## Income Data

In [24]:
income_metrics, income_tpr = run_dataset_experiment("Income")
all_metrics.append(income_metrics)
all_tpr.append(income_tpr)
#plot_dataset_metric_barcharts("Income", income_metrics)

,dataset,partition_fit_data,classifier,roc-auc,pr-auc,accuracy,precision,recall,f1-score
0,Income,id,Logistic Regression,0.9304,0.8558,0.8421,0.7484,0.7731,0.7605
1,Income,id,Random Forest,0.9204,0.8341,0.8336,0.7397,0.7513,0.7454
2,Income,id+ood,Logistic Regression,0.9296,0.8500,0.8406,0.7460,0.7714,0.7585
3,Income,id+ood,Random Forest,0.9203,0.8338,0.8330,0.7410,0.7458,0.7434


,dataset,partition_fit_data,classifier,0.01,0.05,0.1
0,Income,id,Logistic Regression,0.3261,0.5560,0.7141
1,Income,id,Random Forest,0.2709,0.5210,0.6859
2,Income,id+ood,Logistic Regression,0.3329,0.5503,0.7111
3,Income,id+ood,Random Forest,0.2647,0.5302,0.6855


## MVx6 Data

In [25]:
mvx6_metrics, mvx6_tpr = run_dataset_experiment("MVx6")
all_metrics.append(mvx6_metrics)
all_tpr.append(mvx6_tpr)
#plot_dataset_metric_barcharts("MVx6", mvx6_metrics)

,dataset,partition_fit_data,classifier,roc-auc,pr-auc,accuracy,precision,recall,f1-score
0,MVx6,id,Logistic Regression,0.7919,0.8540,0.7825,0.9797,0.5770,0.7263
1,MVx6,id,Random Forest,0.7903,0.8503,0.7753,0.9369,0.5903,0.7242
2,MVx6,id+ood,Logistic Regression,0.7956,0.8557,0.7821,0.9778,0.5773,0.7260
3,MVx6,id+ood,Random Forest,0.7897,0.8500,0.7750,0.9353,0.5909,0.7242


,dataset,partition_fit_data,classifier,0.01,0.05,0.1
0,MVx6,id,Logistic Regression,0.5711,0.5963,0.6192
1,MVx6,id,Random Forest,0.5701,0.5935,0.6145
2,MVx6,id+ood,Logistic Regression,0.5705,0.5981,0.6207
3,MVx6,id+ood,Random Forest,0.5665,0.5948,0.6174


## Diabetes Data

In [26]:
diabetes_metrics, diabetes_tpr = run_dataset_experiment("Diabetes")
all_metrics.append(diabetes_metrics)
all_tpr.append(diabetes_tpr)
#plot_dataset_metric_barcharts("Diabetes", diabetes_metrics)

,dataset,partition_fit_data,classifier,roc-auc,pr-auc,accuracy,precision,recall,f1-score
0,Diabetes,id,Logistic Regression,0.9268,0.3456,0.9556,0.4378,0.2111,0.2849
1,Diabetes,id,Random Forest,0.9344,0.3775,0.9575,0.3846,0.0222,0.0420
2,Diabetes,id+ood,Logistic Regression,0.9271,0.3565,0.9566,0.4630,0.2222,0.3003
3,Diabetes,id+ood,Random Forest,0.9359,0.3639,0.9576,0.4194,0.0289,0.0541


,dataset,partition_fit_data,classifier,0.01,0.05,0.1
0,Diabetes,id,Logistic Regression,0.1844,0.6222,0.8378
1,Diabetes,id,Random Forest,0.1689,0.7022,0.9133
2,Diabetes,id+ood,Logistic Regression,0.1956,0.6089,0.8289
3,Diabetes,id+ood,Random Forest,0.1533,0.6311,0.8933


## California Data

In [27]:
california_metrics, california_tpr = run_dataset_experiment("California")
all_metrics.append(california_metrics)
all_tpr.append(california_tpr)
#plot_dataset_metric_barcharts("California", california_metrics)

,dataset,partition_fit_data,classifier,roc-auc,pr-auc,accuracy,precision,recall,f1-score
0,California,id,Logistic Regression,0.9938,0.9932,0.9583,0.9560,0.9609,0.9584
1,California,id,Random Forest,0.9961,0.9961,0.9658,0.9652,0.9664,0.9658
2,California,id+ood,Logistic Regression,0.9935,0.9935,0.9577,0.9533,0.9625,0.9579
3,California,id+ood,Random Forest,0.9959,0.9960,0.9656,0.9697,0.9612,0.9655


,dataset,partition_fit_data,classifier,0.01,0.05,0.1
0,California,id,Logistic Regression,0.8966,0.9658,0.9913
1,California,id,Random Forest,0.9238,0.9790,0.9955
2,California,id+ood,Logistic Regression,0.8792,0.9641,0.9900
3,California,id+ood,Random Forest,0.9163,0.9761,0.9945


## ACS Accidents Data

In [28]:
acs_metrics, acs_tpr = run_dataset_experiment("ACS Accidents")
all_metrics.append(acs_metrics)
all_tpr.append(acs_tpr)
#plot_dataset_metric_barcharts("ACS Accidents", acs_metrics)

,dataset,partition_fit_data,classifier,roc-auc,pr-auc,accuracy,precision,recall,f1-score
0,ACS Accidents,id,Logistic Regression,0.9998,0.9984,0.997,0.9932,0.9865,0.9899
1,ACS Accidents,id,Random Forest,1.0000,1.0000,1.000,1.0000,1.0000,1.0000
2,ACS Accidents,id+ood,Logistic Regression,1.0000,1.0000,1.000,1.0000,1.0000,1.0000
3,ACS Accidents,id+ood,Random Forest,1.0000,1.0000,1.000,1.0000,1.0000,1.0000


,dataset,partition_fit_data,classifier,0.01,0.05,0.1
0,ACS Accidents,id,Logistic Regression,0.9992,1.0,1.0
1,ACS Accidents,id,Random Forest,1.0000,1.0,1.0
2,ACS Accidents,id+ood,Logistic Regression,1.0000,1.0,1.0
3,ACS Accidents,id+ood,Random Forest,1.0000,1.0,1.0


## All results

In [29]:
metrics_results = pd.concat(all_metrics, axis=0, ignore_index=True)
tpr_results = pd.concat(all_tpr, axis=0, ignore_index=True)

display(metrics_results)
display(tpr_results)

,dataset,partition_fit_data,classifier,roc-auc,pr-auc,accuracy,precision,recall,f1-score
0,Taxi,id,Logistic Regression,0.9297,0.9221,0.8717,0.8632,0.8833,0.8731
1,Taxi,id,Random Forest,0.9746,0.9745,0.9095,0.9212,0.8957,0.9082
2,Taxi,id+ood,Logistic Regression,0.9997,0.9997,0.9907,0.9920,0.9893,0.9907
3,Taxi,id+ood,Random Forest,0.9996,0.9996,0.9920,0.9900,0.9940,0.9920
4,Taxi,id,Logistic Regression,0.9297,0.9221,0.8717,0.8632,0.8833,0.8731
5,Taxi,id,Random Forest,0.9746,0.9745,0.9095,0.9212,0.8957,0.9082
6,Taxi,id+ood,Logistic Regression,0.9997,0.9997,0.9907,0.9920,0.9893,0.9907
7,Taxi,id+ood,Random Forest,0.9996,0.9996,0.9920,0.9900,0.9940,0.9920
8,Electricity,id,Logistic Regression,0.9578,0.9466,0.8904,0.8671,0.9225,0.8939
9,Electricity,id,Random Forest,0.9810,0.9772,0.9280,0.8997,0.9637,0.9306


,dataset,partition_fit_data,classifier,0.01,0.05,0.1
0,Taxi,id,Logistic Regression,0.2493,0.6757,0.8333
1,Taxi,id,Random Forest,0.6600,0.8567,0.9250
2,Taxi,id+ood,Logistic Regression,0.9933,1.0000,1.0000
3,Taxi,id+ood,Random Forest,0.9940,0.9997,1.0000
4,Taxi,id,Logistic Regression,0.2493,0.6757,0.8333
5,Taxi,id,Random Forest,0.6600,0.8567,0.9250
6,Taxi,id+ood,Logistic Regression,0.9933,1.0000,1.0000
7,Taxi,id+ood,Random Forest,0.9940,0.9997,1.0000
8,Electricity,id,Logistic Regression,0.4186,0.7058,0.8532
9,Electricity,id,Random Forest,0.6366,0.8696,0.9567


In [30]:
comparison = metrics_results.pivot_table(
    index=["dataset", "classifier"],
    columns="partition_fit_data",
    values=["roc-auc", "pr-auc", "accuracy", "precision", "recall", "f1-score"],
)
display(comparison)

accuracy         f1-score          pr-auc  \
partition_fit_data                      id  id+ood       id  id+ood      id   
dataset       classifier                                                      
ACS Accidents Logistic Regression   0.9970  1.0000   0.9899  1.0000  0.9984   
              Random Forest         1.0000  1.0000   1.0000  1.0000  1.0000   
California    Logistic Regression   0.9583  0.9577   0.9584  0.9579  0.9932   
              Random Forest         0.9658  0.9656   0.9658  0.9655  0.9961   
Diabetes      Logistic Regression   0.9556  0.9566   0.2849  0.3003  0.3456   
              Random Forest         0.9575  0.9576   0.0420  0.0541  0.3775   
Electricity   Logistic Regression   0.8904  0.8870   0.8939  0.8913  0.9466   
              Random Forest         0.9280  0.9255   0.9306  0.9282  0.9772   
Income        Logistic Regression   0.8421  0.8406   0.7605  0.7585  0.8558   
              Random Forest         0.8336  0.8330   0.7454  0.7434  0.8341   
MVx6          Logistic Regression   0.7825  0.7821   0.7263  0.7260  0.8540   
              Random Forest         0.7753  0.7750   0.7242  0.7242  0.8503   
Taxi          Logistic Regression   0.8717  0.9907   0.8731  0.9907  0.9221   
              Random Forest         0.9095  0.9920   0.9082  0.9920  0.9745   

                                          precision          recall          \
partition_fit_data                 id+ood        id  id+ood      id  id+ood   
dataset       classifier                                                      
ACS Accidents Logistic Regression  1.0000    0.9932  1.0000  0.9865  1.0000   
              Random Forest        1.0000    1.0000  1.0000  1.0000  1.0000   
California    Logistic Regression  0.9935    0.9560  0.9533  0.9609  0.9625   
              Random Forest        0.9960    0.9652  0.9697  0.9664  0.9612   
Diabetes      Logistic Regression  0.3565    0.4378  0.4630  0.2111  0.2222   
              Random Forest        0.3639    0.3846  0.4194  0.0222  0.0289   
Electricity   Logistic Regression  0.9487    0.8671  0.8597  0.9225  0.9255   
              Random Forest        0.9785    0.8997  0.8967  0.9637  0.9621   
Income        Logistic Regression  0.8500    0.7484  0.7460  0.7731  0.7714   
              Random Forest        0.8338    0.7397  0.7410  0.7513  0.7458   
MVx6          Logistic Regression  0.8557    0.9797  0.9778  0.5770  0.5773   
              Random Forest        0.8500    0.9369  0.9353  0.5903  0.5909   
Taxi          Logistic Regression  0.9997    0.8632  0.9920  0.8833  0.9893   
              Random Forest        0.9996    0.9212  0.9900  0.8957  0.9940   

                                  roc-auc          
partition_fit_data                     id  id+ood  
dataset       classifier                           
ACS Accidents Logistic Regression  0.9998  1.0000  
              Random Forest        1.0000  1.0000  
California    Logistic Regression  0.9938  0.9935  
              Random Forest        0.9961  0.9959  
Diabetes      Logistic Regression  0.9268  0.9271  
              Random Forest        0.9344  0.9359  
Electricity   Logistic Regression  0.9578  0.9592  
              Random Forest        0.9810  0.9818  
Income        Logistic Regression  0.9304  0.9296  
              Random Forest        0.9204  0.9203  
MVx6          Logistic Regression  0.7919  0.7956  
              Random Forest        0.7903  0.7897  
Taxi          Logistic Regression  0.9297  0.9997  
              Random Forest        0.9746  0.9996